# Логистическая регрессия и метрики классификации

В этом ноутбуке вы разберёте полный базовый пайплайн бинарной классификации:

- загрузка и первичный анализ данных;
- train/test split;
- обучение логистической регрессии;
- получение вероятностей и классов;
- confusion matrix;
- accuracy, precision, recall, F1;
- ROC-AUC и PR-AUC;
- влияние threshold на качество модели.

Главный фокус практики — **самостоятельная реализация метрик классификации**.

Вам нужно будет реализовать метрики руками через `numpy`, а затем сравнить их с реализациями из `sklearn`.

## 1. Постановка задачи

В задаче бинарной классификации модель должна отнести объект к одному из двух классов:

$$
y \in \{0, 1\}
$$

Класс `1` обычно называют положительным, а класс `0` — отрицательным.

В этом ноутбуке будем использовать датасет `breast_cancer` из `sklearn`.

Задача — по признакам опухоли предсказать класс:

- `0` — malignant;
- `1` — benign.

Наша цель — обучить логистическую регрессию и научиться оценивать качество классификации.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

import matplotlib.pyplot as plt

## 2. Загрузка данных

Загрузим датасет и посмотрим на признаки.

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Размер матрицы признаков:", X.shape)
print("Размер целевой переменной:", y.shape)
print("Классы:", data.target_names)

X.head()

Посмотрим распределение классов.

In [ ]:
class_counts = y.value_counts().sort_index()
class_counts.index = [f"{idx}: {data.target_names[idx]}" for idx in class_counts.index]
class_counts

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(class_counts.index, class_counts.values)
plt.title("Распределение классов")
plt.xlabel("Класс")
plt.ylabel("Количество объектов")
plt.xticks(rotation=15)
plt.show()

## 3. Train/test split

Разделим данные на обучающую и тестовую части.

Так как классы могут быть несбалансированы, используем `stratify=y`, чтобы сохранить долю классов в train и test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nДоля классов в train:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nДоля классов в test:")
print(y_test.value_counts(normalize=True).sort_index())

## 4. Обучение логистической регрессии

Логистическая регрессия сначала считает линейный score:

$$
z = w^Tx + b
$$

Затем применяет сигмоиду:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

И получает вероятность положительного класса:

$$
p = P(y = 1 | x)
$$

Так как логистическая регрессия чувствительна к масштабу признаков, перед моделью добавим `StandardScaler`.

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)

Получим предсказанные классы и вероятности положительного класса.

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Первые 10 предсказанных классов:")
print(y_pred[:10])

print("\nПервые 10 вероятностей положительного класса:")
print(y_proba[:10])

## 5. Confusion matrix

Для бинарной классификации удобно использовать confusion matrix.

|  | Предсказано 1 | Предсказано 0 |
|---|---:|---:|
| Истинный класс 1 | TP | FN |
| Истинный класс 0 | FP | TN |

Где:

- **TP** — модель предсказала 1, истинный класс тоже 1;
- **FP** — модель предсказала 1, истинный класс 0;
- **TN** — модель предсказала 0, истинный класс тоже 0;
- **FN** — модель предсказала 0, истинный класс 1.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
tn, fp, fn, tp = cm.ravel()

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

# Самостоятельная реализация метрик

Ниже начинается основная часть задания.

Вам нужно реализовать метрики самостоятельно.

Важно: в реализациях ниже нельзя использовать готовые функции `accuracy_score`, `precision_score`, `recall_score`, `f1_score` и другие функции из `sklearn.metrics`.

Их можно использовать только в блоке проверки.

## Задание 1. Реализуйте confusion matrix

Напишите функцию `my_confusion_matrix`, которая принимает:

- `y_true` — истинные метки;
- `y_pred` — предсказанные метки.

Функция должна вернуть четыре значения:

```python
tn, fp, fn, tp
```

In [ ]:
def my_confusion_matrix(y_true, y_pred):
    """
    Реализуйте подсчёт TN, FP, FN, TP.

    Параметры:
    y_true: массив истинных меток 0/1
    y_pred: массив предсказанных меток 0/1

    Возвращает:
    tn, fp, fn, tp
    """

    # TODO: переведите y_true и y_pred в numpy-массивы
    # y_true = ...
    # y_pred = ...

    # TODO: посчитайте TN, FP, FN, TP
    # tn = ...
    # fp = ...
    # fn = ...
    # tp = ...

    # return tn, fp, fn, tp

    raise NotImplementedError("Реализуйте функцию my_confusion_matrix")

Проверьте свою реализацию.

In [ ]:
# После реализации этой ячейки ошибка должна исчезнуть

# my_tn, my_fp, my_fn, my_tp = my_confusion_matrix(y_test, y_pred)

# print("Моя реализация:")
# print("TN:", my_tn)
# print("FP:", my_fp)
# print("FN:", my_fn)
# print("TP:", my_tp)

# print("\nsklearn:")
# print("TN:", tn)
# print("FP:", fp)
# print("FN:", fn)
# print("TP:", tp)

# assert (my_tn, my_fp, my_fn, my_tp) == (tn, fp, fn, tp)

## Задание 2. Реализуйте accuracy

Accuracy — доля правильных ответов модели:

$$
Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
$$

In [ ]:
def my_accuracy_score(y_true, y_pred):
    """
    Реализуйте accuracy.

    Возвращает число от 0 до 1.
    """

    # TODO: используйте my_confusion_matrix
    # tn, fp, fn, tp = ...

    # TODO: посчитайте accuracy
    # accuracy = ...

    # return accuracy

    raise NotImplementedError("Реализуйте функцию my_accuracy_score")

In [ ]:
# Проверка

# my_acc = my_accuracy_score(y_test, y_pred)
# sk_acc = accuracy_score(y_test, y_pred)

# print("my accuracy:", my_acc)
# print("sklearn accuracy:", sk_acc)

# assert np.isclose(my_acc, sk_acc)

## Задание 3. Реализуйте precision

Precision показывает, какая доля объектов, предсказанных как положительные, действительно является положительной:

$$
Precision = \frac{TP}{TP + FP}
$$

Precision важен, когда цена ложноположительной ошибки высокая.

In [ ]:
def my_precision_score(y_true, y_pred):
    """
    Реализуйте precision.

    Если TP + FP = 0, верните 0.0.
    """

    # TODO: используйте my_confusion_matrix
    # tn, fp, fn, tp = ...

    # TODO: обработайте случай деления на ноль
    # if ...:
    #     return 0.0

    # TODO: посчитайте precision
    # precision = ...

    # return precision

    raise NotImplementedError("Реализуйте функцию my_precision_score")

In [ ]:
# Проверка

# my_precision = my_precision_score(y_test, y_pred)
# sk_precision = precision_score(y_test, y_pred)

# print("my precision:", my_precision)
# print("sklearn precision:", sk_precision)

# assert np.isclose(my_precision, sk_precision)

## Задание 4. Реализуйте recall

Recall показывает, какую долю всех положительных объектов модель смогла найти:

$$
Recall = \frac{TP}{TP + FN}
$$

Recall важен, когда цена ложноотрицательной ошибки высокая.

In [ ]:
def my_recall_score(y_true, y_pred):
    """
    Реализуйте recall.

    Если TP + FN = 0, верните 0.0.
    """

    # TODO: используйте my_confusion_matrix
    # tn, fp, fn, tp = ...

    # TODO: обработайте случай деления на ноль
    # if ...:
    #     return 0.0

    # TODO: посчитайте recall
    # recall = ...

    # return recall

    raise NotImplementedError("Реализуйте функцию my_recall_score")

In [ ]:
# Проверка

# my_recall = my_recall_score(y_test, y_pred)
# sk_recall = recall_score(y_test, y_pred)

# print("my recall:", my_recall)
# print("sklearn recall:", sk_recall)

# assert np.isclose(my_recall, sk_recall)

## Задание 5. Реализуйте F1-score

F1-score объединяет precision и recall:

$$
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
$$

F1 будет высоким только тогда, когда и precision, и recall достаточно высокие.

In [ ]:
def my_f1_score(y_true, y_pred):
    """
    Реализуйте F1-score.

    Если precision + recall = 0, верните 0.0.
    """

    # TODO: используйте my_precision_score и my_recall_score
    # precision = ...
    # recall = ...

    # TODO: обработайте случай деления на ноль
    # if ...:
    #     return 0.0

    # TODO: посчитайте F1
    # f1 = ...

    # return f1

    raise NotImplementedError("Реализуйте функцию my_f1_score")

In [ ]:
# Проверка

# my_f1 = my_f1_score(y_test, y_pred)
# sk_f1 = f1_score(y_test, y_pred)

# print("my F1:", my_f1)
# print("sklearn F1:", sk_f1)

# assert np.isclose(my_f1, sk_f1)

## 6. Сравнение всех метрик

После реализации всех функций соберите таблицу со своими метриками и метриками из `sklearn`.

In [ ]:
# Раскомментируйте после реализации функций

# metrics_comparison = pd.DataFrame({
#     "my implementation": [
#         my_accuracy_score(y_test, y_pred),
#         my_precision_score(y_test, y_pred),
#         my_recall_score(y_test, y_pred),
#         my_f1_score(y_test, y_pred),
#     ],
#     "sklearn": [
#         accuracy_score(y_test, y_pred),
#         precision_score(y_test, y_pred),
#         recall_score(y_test, y_pred),
#         f1_score(y_test, y_pred),
#     ]
# }, index=["accuracy", "precision", "recall", "f1"])

# metrics_comparison

## 7. Threshold и вероятности

Логистическая регрессия выдаёт вероятность положительного класса:

$$
P(y = 1 | x)
$$

Чтобы получить класс, используется threshold.

По умолчанию:

$$
threshold = 0.5
$$

Если вероятность больше или равна 0.5, предсказываем класс 1. Иначе — класс 0.

In [ ]:
def predict_with_threshold(y_proba, threshold=0.5):
    return (y_proba >= threshold).astype(int)

Посмотрим, как меняются метрики при разных threshold.

После реализации своих метрик можно заменить функции sklearn на свои.

In [ ]:
thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]

rows = []

for threshold in thresholds:
    preds = predict_with_threshold(y_proba, threshold)

    rows.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0)
    })

threshold_report = pd.DataFrame(rows)
threshold_report

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(threshold_report["threshold"], threshold_report["precision"], marker="o", label="precision")
plt.plot(threshold_report["threshold"], threshold_report["recall"], marker="o", label="recall")
plt.plot(threshold_report["threshold"], threshold_report["f1"], marker="o", label="f1")

plt.title("Метрики при разных threshold")
plt.xlabel("threshold")
plt.ylabel("metric value")
plt.ylim(0, 1.05)
plt.legend()
plt.show()

## 8. ROC-AUC и PR-AUC

ROC-AUC и PR-AUC считаются не по готовым классам, а по score или вероятностям.

ROC-AUC измеряет качество ранжирования объектов:

- `1.0` — идеальное ранжирование;
- `0.5` — случайное ранжирование;
- меньше `0.5` — хуже случайного.

PR-AUC особенно полезна при дисбалансе классов.

In [ ]:
roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC curve")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
plt.show()

In [ ]:
precision_values, recall_values, pr_thresholds = precision_recall_curve(y_test, y_proba)

plt.figure(figsize=(7, 5))
plt.plot(recall_values, precision_values, label=f"PR-AUC = {pr_auc:.3f}")
plt.title("Precision-Recall curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.show()

# Итоговые задания

Выполните задания ниже.

## Задание 1

Реализуйте функции:

- `my_confusion_matrix`;
- `my_accuracy_score`;
- `my_precision_score`;
- `my_recall_score`;
- `my_f1_score`.

## Задание 2

Сравните свои реализации с функциями из `sklearn`.

Результаты должны совпадать или отличаться только на очень маленькую численную погрешность.

## Задание 3

Постройте таблицу метрик при threshold:

- 0.1;
- 0.3;
- 0.5;
- 0.7;
- 0.9.

## Задание 4

Ответьте на вопросы:

1. Как меняется precision при увеличении threshold?
2. Как меняется recall при увеличении threshold?
3. Почему precision и recall часто движутся в разные стороны?
4. Какой threshold вы бы выбрали для этой задачи и почему?
5. Почему accuracy не всегда достаточно для оценки классификатора?

In [ ]:
# Место для ваших выводов

# 1.
# 2.
# 3.
# 4.
# 5.

## Критерии оценки

Всего: **60 баллов**.

- обучение логистической регрессии и получение вероятностей — 10 баллов;
- реализация confusion matrix — 10 баллов;
- реализация accuracy, precision, recall и F1 — 20 баллов;
- сравнение своих метрик со sklearn — 10 баллов;
- анализ влияния threshold — 5 баллов;
- итоговые выводы — 5 баллов.

## Что важно запомнить

1. Логистическая регрессия решает задачу классификации, хотя называется регрессией.
2. Модель выдаёт вероятность положительного класса.
3. Чтобы получить класс, нужно выбрать threshold.
4. Accuracy может быть плохой метрикой при дисбалансе классов.
5. Precision важен, когда дорогие FP-ошибки.
6. Recall важен, когда дорогие FN-ошибки.
7. F1 объединяет precision и recall.
8. ROC-AUC и PR-AUC считаются по вероятностям или score, а не по готовым классам.